# Частина 2: Аналіз датасету Individual Household Electric Power Consumption

## Завдання 1: Зчитування та очищення даних
Завантаження датасету, обробка пропущених значень та конвертація стовпців часу і дати у відповідні формати.

In [2]:
import pandas as pd

df = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)
df.dropna(inplace=True)
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time
df['Global_active_power'] = pd.to_numeric(df['Global_active_power'])
df['Global_intensity'] = pd.to_numeric(df['Global_intensity'])

display(df.head())

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Завдання 2: Формування вибірок
Реалізація окремих функцій для фільтрації записів за потужністю, силою струму, показниками груп споживання (Sub_metering) та часом.

In [3]:
def filter_power_over_5():
    return df[df['Global_active_power'] > 5.0]

def filter_intensity_and_sub_metering():
    cond_1 = (df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)
    cond_2 = df['Sub_metering_2'] > df['Sub_metering_3']
    return df[cond_1 & cond_2]

def random_sample_means():
    sample = df.sample(n=500000, replace=False)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def filter_evening_high_consumption():
    time_limit = pd.to_datetime('18:00:00', format='%H:%M:%S').time()
    cond_1 = df['Time'] > time_limit
    cond_2 = df['Global_active_power'] > 6.0
    cond_3 = (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])
    
    filtered = df[cond_1 & cond_2 & cond_3]
    
    half_idx = len(filtered) // 2
    first_half = filtered.iloc[:half_idx].iloc[2::3]
    second_half = filtered.iloc[half_idx:].iloc[3::4]
    
    return pd.concat([first_half, second_half])

display(filter_power_over_5().head())
display(filter_intensity_and_sub_metering().head())
display(random_sample_means())
display(filter_evening_high_consumption().head())

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,2006-12-16,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,2006-12-16,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,2006-12-16,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,2006-12-17,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,2006-12-17,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,2006-12-17,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,2006-12-17,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


Sub_metering_1    1.110844
Sub_metering_2    1.289034
Sub_metering_3    6.460220
dtype: float64

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
43,2006-12-16,18:07:00,6.474,0.144,231.85,27.8,0.0,37.0,16.0
3007,2006-12-18,19:31:00,6.158,0.442,229.08,27.0,0.0,36.0,0.0
17497,2006-12-28,21:01:00,7.062,0.270,235.76,30.2,2.0,65.0,17.0
17500,2006-12-28,21:04:00,7.376,0.238,234.67,31.4,1.0,72.0,17.0
17503,2006-12-28,21:07:00,7.248,0.000,235.34,30.8,1.0,72.0,17.0


## Завдання 3: Профілювання часу виконання
Аналіз часових витрат на виконання кожної з процедур вибірки за допомогою модуля `timeit`.

In [4]:
import timeit

t1 = timeit.timeit(filter_power_over_5, number=10)
t2 = timeit.timeit(filter_intensity_and_sub_metering, number=10)
t3 = timeit.timeit(random_sample_means, number=10)
t4 = timeit.timeit(filter_evening_high_consumption, number=10)

print(f"Функція 1: {t1:.4f} с")
print(f"Функція 2: {t2:.4f} с")
print(f"Функція 3: {t3:.4f} с")
print(f"Функція 4: {t4:.4f} с")

Функція 1: 0.0597 с
Функція 2: 0.1156 с
Функція 3: 2.6640 с
Функція 4: 0.8609 с


## Завдання 4: Статистичний аналіз та кодування
Нормування та стандартизація атрибутів, підрахунок коефіцієнтів кореляції Пірсона та Спірмена, а також One Hot Encoding для категоріального атрибута.

In [6]:
cols = ['Global_active_power', 'Global_intensity']

df_normalized = (df[cols] - df[cols].min()) / (df[cols].max() - df[cols].min())
df_standardized = (df[cols] - df[cols].mean()) / df[cols].std()

pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Коефіцієнт Пірсона: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_corr:.4f}")

df['Month'] = df['Date'].dt.month
df_encoded = pd.get_dummies(df, columns=['Month'], drop_first=False)

display(df_normalized.head())
display(df_standardized.head())
display(df_encoded.head())

Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


,Global_active_power,Global_intensity
0,0.374796,0.377593
1,0.478363,0.473029
2,0.479631,0.473029
3,0.480898,0.473029
4,0.325005,0.323651


,Global_active_power,Global_intensity
0,2.955076,3.098788
1,4.037084,4.133799
2,4.050325,4.133799
3,4.063566,4.133799
4,2.434881,2.513781


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Month_1,...,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9,Month_10,Month_11,Month_12
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,False,...,False,False,False,False,False,False,False,False,False,True
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,False,...,False,False,False,False,False,False,False,False,False,True
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,False,...,False,False,False,False,False,False,False,False,False,True
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,False,...,False,False,False,False,False,False,False,False,False,True
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,False,...,False,False,False,False,False,False,False,False,False,True
